# Extract AMI Phenotype

In [ ]:
import pandas as pd
import numpy as np
import os
import sys
pd.set_option('display.max_rows', 500)

In [ ]:


root_path = os.path.dirname(os.path.abspath(os.path.dirname('__file__')))
sys.path.insert(0, root_path)

In [ ]:
from env.parameters import P


In [ ]:

from phenotyping.codebase_phenotyping import (
pheno_all_evdt_extractor,
pheno_rank_and_filter,
pheno_keep_one_row,
clean_biobank_ado_pheno,
join_gp_hes_biobank_single_row_dfs,
join_single_row_dfs
)
from util.parquet_maker import dask_to_parquet
from util.general_utils import field_availability_check, list_field_instances
from util.dask_utils import import_field_from_dask
from util.parquet_maker import dask_to_parquet
import dask.dataframe as dd


# Load main files

In [ ]:
# cohort
df_cohort = pd.read_csv(f'''{P.output_csv_path}/cohort_2_advanced.csv''')
df_cohort.head()

In [ ]:
dd_gp = dd.read_parquet(f'''{P.output_parquet_path}/gp_clinical_curated''')
dd_gp.head()

In [ ]:
df_gp = dd_gp[["eid", "read_curated", "evdt_gp"]].compute()
df_gp.dtypes

In [ ]:
df_hes = pd.read_csv(f'''{P.output_csv_path}/hesin_diag_curated.csv''')
df_hes

# AMI



In [ ]:
codelist_in = pd.read_csv(f'''{P.codelist_path}/curated/ami_updated.csv''')
codelist_in.head()

In [ ]:
assign_pheno_name = "ami"

In [ ]:
codelist_in["vocab"].value_counts(dropna=False)

In [ ]:
codelist_in["pheno_name"].value_counts(dropna=False)

## AMI in GP

In [ ]:
out_gp_long = pheno_all_evdt_extractor(df_data_clean=df_gp,
                                             col_evdt_data="evdt_gp",
                                             col_code_data="read_curated",
                                             df_codelist=codelist_in,
                                             col_code_codelist="code_clean",
                                             col_vocab_codelist="vocab",
                                             use_vocab="Read2",
                                             col_assign_pheno_name="pheno",
                                             assign_pheno_name=assign_pheno_name,
                                             col_codelist_multicategory=None,
                                             join_type="read2")

In [ ]:
out_gp_long

In [ ]:
out_gp_long = out_gp_long[['eid', 'evdt_gp', 'pheno']]
out_gp_long.head(5)

In [ ]:
out_gp_long_ranked = pheno_rank_and_filter(out_gp_long, col_evdt="evdt_gp", col_eid="eid", earliest_ranks_1=True)
out_gp_long_ranked.head(10)


In [ ]:
# Use this later for joining with HES (and ADO, if any)
out_gp_single_row = pheno_keep_one_row(df_ranked_long=out_gp_long_ranked,
                                       col_row_number="row_number")
out_gp_single_row.head(10)

In [ ]:
 out_gp_single_row['eid'].count() == out_gp_single_row['eid'].nunique()

## AMI in HES

In [ ]:
df_hes_long =pheno_all_evdt_extractor(df_data_clean=df_hes,
                                             col_evdt_data="epistart",
                                             col_code_data="diag_icd10",
                                             df_codelist=codelist_in,
                                             col_code_codelist="code_clean",
                                             col_vocab_codelist="vocab",
                                             use_vocab="ICD10",
                                             col_assign_pheno_name="pheno",
                                             assign_pheno_name=assign_pheno_name,
                                             col_codelist_multicategory=None,
                                             join_type="icd10")
df_hes_long.head()

In [ ]:
df_hes_long = df_hes_long[['eid', 'epistart', 'pheno']]
df_hes_long.head(5)

In [ ]:
out_hes_long_ranked = pheno_rank_and_filter(df_hes_long, col_evdt="epistart", col_eid="eid", earliest_ranks_1=True)
out_hes_long_ranked.head(10)

In [ ]:
# Use this later for joining with GP (and ADO, if any)
out_hes_single_row = pheno_keep_one_row(df_ranked_long=out_hes_long_ranked,
                                       col_row_number="row_number")
out_hes_single_row.head(10)

# Make a single GP + HES pheno and save

In [ ]:
df_gp_hes = join_gp_hes_biobank_single_row_dfs(pheno_gp=out_gp_single_row, pheno_hes=out_hes_single_row,
                                   col_evdt_gp="evdt_gp", col_evdt_hes="epistart", pheno_name=assign_pheno_name, gp_and_hes_only=True, keep_minimum=True,col_eid="eid")

In [ ]:
df_gp_hes = df_gp_hes[['eid', f'''evdt_{assign_pheno_name}''']]
#df_out[f'''pheno_{assign_pheno_name}'''] = 1
df_gp_hes.head()

In [ ]:
assign_pheno_name

In [ ]:
# Save
df_gp_hes.to_csv(f'''{P.output_phenotypes_csv_path}/incident_{assign_pheno_name}_gp_hes.csv''', index=False)


In [ ]:
df_gp_hes = pd.read_csv(f'''{P.output_phenotypes_csv_path}/incident_{assign_pheno_name}_gp_hes.csv''')
df_gp_hes.head()

In [ ]:
df_gp_hes[f'''evdt_{assign_pheno_name}'''] = pd.to_datetime(df_gp_hes[f'''evdt_{assign_pheno_name}'''])


# AMI in Biobank ADO field

In [ ]:
# core_all
dd_all = dd.read_parquet(f'''{P.output_parquet_path}/{P.core_all_name}''')
dd_all.head()

In [ ]:
cols_all = dd_all.columns

In [ ]:
#ADO MI
search_term = "42000"
list_term = list_field_instances(col_list=cols_all, search_for_field=search_term)
print(list_term)
dd_ado_mi = import_field_from_dask(dd_in= dd_all, field_name=search_term, eid_col="eid")
# to pandas
df_ado_mi = dd_ado_mi.compute()
print(df_ado_mi.dtypes)
print(df_ado_mi.head())

In [ ]:
df_ado_mi["eid"] = df_ado_mi["eid"].astype("int64")
df_ado_mi["42000-0.0"] = pd.to_datetime(df_ado_mi["42000-0.0"])
print(df_ado_mi.dtypes)
print(df_ado_mi.head())

In [ ]:
#ADO AMI source
search_term = "42001"
list_term = list_field_instances(col_list=cols_all, search_for_field=search_term)
print(list_term)
dd_ado_mi_source = import_field_from_dask(dd_in= dd_all, field_name=search_term, eid_col="eid")
# to pandas
df_ado_mi_source = dd_ado_mi_source.compute()
print(df_ado_mi_source.dtypes)
print(df_ado_mi_source.head())

In [ ]:
df_ado_mi_source["eid"] = df_ado_mi_source["eid"].astype("int64")
# Source of asthma is nullable
df_ado_mi_source["42001-0.0"] = df_ado_mi_source["42001-0.0"].astype("Int64")
print(df_ado_mi_source.dtypes)
print(df_ado_mi_source.head())

In [ ]:
df_ado_j = pd.merge(df_ado_mi, df_ado_mi_source, on="eid", how="left")
df_ado_j.head()

In [ ]:
assign_pheno_name

In [ ]:
pheno_ado = clean_biobank_ado_pheno(df_ado_j,
                                           col_evdt="42000-0.0",
                                           col_source="42001-0.0",
                                           df_cohort=df_cohort,
                                           pheno_name=assign_pheno_name,
                                           col_dob="dob",
                                           col_dod= "dod",
                                           col_eid="eid")
pheno_ado.head()

In [ ]:
pheno_ado['eid'].count() == pheno_ado['eid'].nunique()

In [ ]:
df_gp_hes.head()

In [ ]:
df_gp_hes.dtypes

In [ ]:
df_gp_hes = df_gp_hes.rename(columns={f'''evdt_{assign_pheno_name}''': f'''evdt_gp_hes_{assign_pheno_name}'''})


In [ ]:
df_all = join_single_row_dfs(pheno_1= df_gp_hes,
                               pheno_2= pheno_ado,
                               col_evdt_1= f'''evdt_gp_hes_{assign_pheno_name}''',
                               col_evdt_2= f'''evdt_ado_{assign_pheno_name}''',
                               pheno_name= assign_pheno_name,
                               keep_extra_cols_1=[],
                               keep_extra_cols_2=[f'''source_ado_{assign_pheno_name}'''],
                               keep_minimum=True
                               )

In [ ]:
df_all.head()

In [ ]:
df_all.value_counts(subset=[f'''source_ado_{assign_pheno_name}'''], dropna=False)


In [ ]:
df_out = df_all[['eid', f'''evdt_{assign_pheno_name}''', f'''source_ado_{assign_pheno_name}''']]
#df_out['pheno_asthma'] = 1
df_out.head()

In [ ]:
df_out.dtypes


In [ ]:
df_out.shape


In [ ]:

# Save
df_out.to_csv(f'''{P.output_phenotypes_csv_path}/incident_{assign_pheno_name}_gp_hes_ado.csv''', index=False)
